<a href="https://colab.research.google.com/github/NadhaKaleel/northstar-analytics/blob/main/notebooks/01_sql_in_r.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
# Install necessary packages
install.packages(c('sqldf', 'dplyr', 'ggplot2'))

# Load libraries
library(sqldf)
library(dplyr)
library(ggplot2)

# 2. Define the path (Since they are uploaded to the sidebar, use './')
path <- "./"

# 3. Load files
# Using stringsAsFactors=FALSE is a good practice for older R versions
orders     <- read.csv(paste0(path, "orders.csv"))
deliveries <- read.csv(paste0(path, "deliveries.csv"))
customers  <- read.csv(paste0(path, "customers.csv"))
drivers    <- read.csv(paste0(path, "drivers.csv"))
vehicles   <- read.csv(paste0(path, "vehicles.csv"))
hubs       <- read.csv(paste0(path, "hubs.csv"))
incidents  <- read.csv(paste0(path, "incidents.csv"))
complaints <- read.csv(paste0(path, "complaints.csv"))
app_events <- read.csv(paste0(path, "app_events.csv"))

result1 <- sqldf(
 "SELECT d.hub_id, h.hub_name, h.zone, d.delivery_status,
         COUNT(*) AS count,
         ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (PARTITION BY d.hub_id), 2) AS pct
  FROM deliveries d JOIN hubs h ON d.hub_id = h.hub_id
  GROUP BY d.hub_id, h.hub_name, h.zone, d.delivery_status
  ORDER BY d.hub_id, count DESC"
)
# Filter for failed deliveries only
print(result1[result1$delivery_status == 'Failed', ])

del_filt <- deliveries[deliveries$manual_route_override_count > 0, ]

result2 <- sqldf(
 "SELECT d.driver_id, dr.base_zone, dr.employment_type,
         COUNT(*) AS total_deliveries,
         SUM(d.manual_route_override_count) AS total_overrides,
         ROUND(AVG(d.manual_route_override_count), 2) AS avg_overrides,
         ROUND(AVG(d.customer_rating_post_delivery), 2) AS avg_rating,
         ROUND(SUM(CASE WHEN d.delivery_status='Failed' THEN 1.0 ELSE 0 END)
               / COUNT(*) * 100, 1) AS fail_rate_pct
  FROM del_filt d JOIN drivers dr ON d.driver_id = dr.driver_id
  GROUP BY d.driver_id, dr.base_zone, dr.employment_type
  HAVING total_deliveries >= 3
  ORDER BY avg_overrides DESC LIMIT 8"
)
print(result2)

result3 <- sqldf(
 "SELECT c.customer_id, c.home_zone, c.customer_type,
         ROUND(c.loyalty_score, 1)              AS loyalty_score,
         COUNT(DISTINCT cp.complaint_id)         AS complaint_count,
         COUNT(DISTINCT CASE WHEN d.delivery_status='Failed'
               THEN d.delivery_id END)           AS failed_deliveries,
         ROUND(SUM(cp.compensation_amount), 2)   AS total_compensation
  FROM customers c
  LEFT JOIN complaints cp ON c.customer_id = cp.customer_id
  LEFT JOIN orders o      ON c.customer_id = o.customer_id
  LEFT JOIN deliveries d  ON o.order_id    = d.order_id
  GROUP BY c.customer_id, c.home_zone, c.customer_type, c.loyalty_score
  HAVING complaint_count >= 2
  ORDER BY complaint_count DESC, total_compensation DESC LIMIT 8"
)
print(result3)

result4 <- sqldf(
 "SELECT
    CASE WHEN v.battery_health_pct >= 80 THEN 'Good (80-100%)'
         WHEN v.battery_health_pct >= 60 THEN 'Fair (60-79%)'
         ELSE 'Poor (<60%)'
    END AS battery_band,
    COUNT(DISTINCT v.vehicle_id)  AS vehicles_in_band,
    COUNT(i.incident_id)          AS total_incidents,
    ROUND(COUNT(i.incident_id)*1.0 / COUNT(DISTINCT d.delivery_id), 3) AS incidents_per_delivery,
    ROUND(AVG(CAST(i.resolved_hours AS REAL)), 2) AS avg_resolve_hrs
  FROM vehicles v
  JOIN deliveries d  ON v.vehicle_id    = d.vehicle_id
  LEFT JOIN incidents i ON d.delivery_id = i.delivery_id
  GROUP BY battery_band ORDER BY avg_resolve_hrs DESC"
)
print(result4)

result5 <- sqldf(
 "SELECT o.pickup_zone, o.service_type,
         COUNT(o.order_id) AS order_count,
         ROUND(AVG(o.order_value), 2) AS avg_order_value,
         ROUND(AVG(d.fuel_or_charge_cost), 2) AS avg_fuel_cost,
         ROUND(AVG(o.order_value - d.fuel_or_charge_cost), 2) AS avg_net_margin,
         ROUND(AVG(d.route_distance_km), 2) AS avg_distance_km
  FROM orders o JOIN deliveries d ON o.order_id = d.order_id
  GROUP BY o.pickup_zone, o.service_type
  ORDER BY avg_net_margin ASC LIMIT 8"
)
print(result5)

Installing packages into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)



   hub_id       hub_name      zone delivery_status count   pct
3     H01 North Exchange     North          Failed    17 12.50
6     H02     South Link     South          Failed    10  9.43
9     H03      East Dock      East          Failed    11  9.24
12    H04      West Gate      West          Failed    16 12.60
15    H05   Central Core   Central          Failed    23 20.00
18    H06    Airport Hub   Airport          Failed    15 14.42
21    H07  Riverside Hub Riverside          Failed    14 12.17
23    H08  Midtown Relay   Central          Failed    26 20.31
  driver_id base_zone employment_type total_deliveries total_overrides
1      D127   CENTRAL        FullTime                6              17
2      D073      East        FullTime                3               8
3      D074   Central        FullTime                3               8
4      D028     North        FullTime                5              13
5      D139     South        FullTime                4              10
6      